# L13c: The Model Context Protocol
In this lecture, we introduce the Model Context Protocol (MCP), an open standard that connects language model applications to tools. We look at the roles in an MCP system, the messages of a tool call, and the checks that keep a tool call safe.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Describe the roles in an MCP system:__ A host runs a language model and connects to servers, which offer tools. We'll define the host, the client, and the server.
> * __Read the messages of a tool call:__ A client and a server exchange JSON-RPC messages. We'll follow the messages that discover a server, list its tools, and call a tool.
> * __Identify the trust boundaries of a tool call:__ A tool runs code on the user's computer, and the language model writes its arguments. We'll look at what the host and the server each check.

Let's get started!
___

## Examples
We will use the following example to illustrate the key ideas of this lecture:

> [▶ Build an MCP server for the National Weather Service API](CHEME-5800-L13c-Example-MCP-NWS-Fall-2026.ipynb). In this example, we wrap our software development kit (SDK) for the National Weather Service API in an MCP server with two tools. We call the tools as a host would, and trigger both kinds of errors.

___

## Language Models and Tools
A __large language model (LLM)__ is a neural network that reads and writes text. On its own, an LLM cannot look up today's weather or run our Julia code. An application gives an LLM these abilities through __tools__, functions that the application runs when the model asks for them.

In a __tool call__, the model answers a question with the name of a tool and its arguments. The application runs the tool, and the model writes its answer from the result.

The __Model Context Protocol (MCP)__ is an open standard that connects applications to tools. Without a standard, $M$ applications and $N$ tool sources need up to $M\times{N}$ custom integrations. With MCP, each side implements the protocol once, for $M+N$ implementations.

Anthropic released MCP in November 2024, and donated it to the Agentic AI Foundation of the Linux Foundation in December 2025. We use [version `2026-07-28` of the MCP specification](https://modelcontextprotocol.io/specification/2026-07-28).

___

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>

## Hosts, Clients, and Servers
MCP defines three roles. The figure shows them with the two servers that we build this week:

<p align="center">
    <img class="course-diagram" src="figs/Fig-MCP-Architecture/Fig-MCP-Architecture.svg" width="1100" style="max-width:100%; height:auto;" alt="A User box on the left is joined by a two-way arrow to a large blue box labeled Host. Inside the host, a Language model box is joined by gray lines to two Client boxes. A blue two-way arrow labeled MCP messages joins the upper client to a Weather server box, and another joins the lower client to a FRED server box. A dark two-way arrow labeled HTTPS joins the weather server to an NWS API box, and another joins the FRED server to a FRED API box. Notes below say that the host is a chat app, a coding assistant, or our notebook; the servers are our Julia code, started by the host; and the APIs are RESTful web services."/>
</p>

__The roles in an MCP system.__ The model talks only to the host. Each server calls its web API over HTTPS.

The __host__ is the application that the user works with, e.g., a chat app, a coding assistant, or our notebook. The host runs the model, starts the servers, and decides whether each tool call goes ahead. Inside the host, each __client__ talks to one __server__, a separate program that offers tools.

A __transport__ carries the messages between a client and a server. Our servers use the __stdio__ transport: the host writes to the __standard input__ of the server, and reads from its __standard output__, the two text streams that every program has. MCP also defines a __Streamable HTTP__ transport, for a server that runs as a web service.

___

## MCP Messages
MCP messages use __JSON-RPC 2.0__, a JSON format for asking another program to run a function. Each MCP __request__ has four fields: `jsonrpc` (always `"2.0"`), an `id`, a `method`, e.g., `"tools/list"`, and its `params`. The __response__ carries the same `id`, and holds either a `result` or an `error`.

A __notification__ is a message with no `id`, and it gets no response. On the stdio transport, each message is one line of JSON text, and the server writes nothing else to its standard output. Julia writes its log messages to a third stream, the __standard error__.

An `error` holds a numeric `code` and a `message`. Our servers use six codes:

| Code | Meaning |
| --- | --- |
| `-32700` | Parse error: the message is not valid JSON |
| `-32600` | Invalid request: the message is not a JSON-RPC 2.0 request, e.g., its `jsonrpc` field is missing |
| `-32601` | Method not found: the server has no method with this name |
| `-32602` | Invalid params, e.g., the request names a tool that the server does not have |
| `-32603` | Internal error: the server failed while it processed the request |
| `-32022` | Unsupported protocol version |

Version `2026-07-28` of MCP is __stateless__, like a RESTful web service. Every request carries a `_meta` object in its `params`, with the protocol version and the name and capabilities of the client. The server checks the version on every request.

Earlier versions opened each connection with an `initialize` request, which has no `_meta` object. Our servers reject that request with error `-32022`.

### Discover a Server
The [messages](messages) folder holds the messages that our notebook exchanged with the weather server, one JSON file per message, numbered in order. A client starts with a `server/discover` request, [1-discover-request.json](messages/1-discover-request.json), and the server answers with [1-discover-response.json](messages/1-discover-response.json).

__What do we see?__ The `result` lists the supported protocol versions, says that the server offers tools, and gives `instructions` for the model. The `ttlMs` field lets a client reuse the result for one hour. Every result also names the server in its `_meta` object.

### List the Tools
A `tools/list` request returns the `name`, the `description`, and the `inputSchema` of each tool. The model reads the description to decide when to use a tool. The input schema is a __JSON Schema__, a JSON document that lists the arguments of the tool, their types, and which ones are required.

In [2-list-response.json](messages/2-list-response.json), the answer to [2-list-request.json](messages/2-list-request.json), the `get_hourly_forecast` tool takes two required numbers, `latitude` and `longitude`, and an optional integer, `hours`, from `1` to `48`.

### Call a Tool
A `tools/call` request names a tool and gives its arguments. [3-call-request.json](messages/3-call-request.json) asks the `get_hourly_forecast` tool for the forecast of the next hour in Ithaca, NY, and the server answers with [3-call-response.json](messages/3-call-response.json).

__What do we see?__ The `content` array holds the result as text for the model, and `structuredContent` holds the same result as JSON data for a program. The `isError` flag is `false`: the tool ran without an error.

The figure follows one tool call, from the question of the user to the answer of the model:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>

<p align="center">
    <img class="course-diagram" src="figs/Fig-MCP-ToolCall/Fig-MCP-ToolCall.svg" width="1000" style="max-width:100%; height:auto;" alt="A sequence diagram with four dashed lifelines: Language model, Host, MCP server, and Web API. The user asks the host: Will it rain in Ithaca tonight? Step 1, a gray arrow from the host to the model: question and tool list. Step 2, a gray arrow from the model to the host: tool name and arguments. Step 3, a blue arrow from the host to the MCP server: tools/call request. Step 4, a dark arrow from the server to the web API: HTTPS GET. Step 5, a dark arrow back to the server: JSON forecast. Step 6, a blue arrow from the server to the host: tools/call result. Step 7, a gray arrow from the host to the model: tool result. Step 8, a gray arrow from the model to the host: answer. The host shows the answer to the user."/>
</p>

__One tool call.__ Gray arrows stay inside the host, blue arrows are MCP messages, and dark arrows are HTTPS calls to the web API. The weather server makes two of these calls.

### Two Kinds of Errors
A __protocol error__ means that the request itself was wrong. [4-unknown-tool-request.json](messages/4-unknown-tool-request.json) asks for a `get_snowfall` tool that the server does not have, and the server answers with an `error` and no `result`, [4-unknown-tool-response.json](messages/4-unknown-tool-response.json).

A __tool execution error__ means that the tool ran, but failed. [5-paris-request.json](messages/5-paris-request.json) asks for the forecast in Paris, France, which the weather service does not cover. The server answers with a normal `result` that has `isError` set to `true`, [5-paris-response.json](messages/5-paris-response.json).

A protocol error usually points to a bug in the host or in the server. A tool execution error goes back to the model, which can read the reason and try other arguments.

### The Server Loop
Our servers run one loop, which reads one message and writes at most one response:

__Initialize__: Given a set of tools $\mathcal{T}$, each with a name, a description, an input schema, and a handler (the function that runs the tool), and the protocol version $v\gets$ `2026-07-28`.

While the standard input is open __do__:
1. Read the next line $\ell$ that is not blank, and parse it: $m\gets\texttt{parse}(\ell)$.
2. Choose the response $r$:
    - If $\ell$ is not valid JSON __then__ $r\gets$ a parse error, `-32700`.
    - Otherwise, if $m$ is not a JSON-RPC 2.0 message __then__ $r\gets$ an invalid-request error, `-32600`.
    - Otherwise, if $m$ has no `id` __then__ $r\gets\texttt{nothing}$ (notification).
    - Otherwise, if the `_meta` object of $m$ does not name $v$ __then__ $r\gets$ an unsupported-version error, `-32022`.
    - Otherwise, if the method of $m$ is `server/discover` __then__ $r\gets$ the versions, capabilities, and identity of the server.
    - Otherwise, if the method is `tools/list` __then__ $r\gets$ the name, description, and input schema of each tool in $\mathcal{T}$.
    - Otherwise, if the method is `tools/call` and $\mathcal{T}$ holds the named tool __then__ run its handler, and set $r\gets$ its result, with `isError` set to `true` if the handler fails (tool execution error).
    - Otherwise, if the method is `tools/call` __then__ $r\gets$ an invalid-params error, `-32602` (unknown tool).
    - Otherwise, $r\gets$ a method-not-found error, `-32601`.
3. If $r\neq\texttt{nothing}$ __then__ write $r$ as one line of JSON text to the standard output.

__Output__: None. The loop ends when the client closes the standard input of the server.

[The `process_message(...)` function](src/Server.jl) carries out step 2 for one line, and [the `serve(...)` function](src/Server.jl) runs the loop.

___

## Trust and Security
A tool call crosses three __trust boundaries__: from the model to the host, from the host to the server, and from the server to a web API. At the first boundary, the model only asks for a call, and the host decides whether the call goes ahead, often by asking the user.

At the second boundary, a stdio server runs with our permissions. We should install a server only when we trust its code. Text from a server can also tell the model to do something that the user did not ask for, an attack called __prompt injection__.

At the third boundary, the server checks every argument that the model wrote before it puts the argument into a URL. An API key stays in the server, and never appears in an MCP message.

Our servers only read public data. A tool that changes data, e.g., one that places an order, should need the approval of the user for each call.

___

## Summary
In this lecture, we introduced the Model Context Protocol, the messages of a tool call, and the checks that keep a tool call safe.

> __Key Takeaways:__
>
> * __Hosts, clients, and servers:__ A host runs a language model and one client for each MCP server, and each server offers tools. The model never talks to a server, because every tool call passes through the host.
> * __Discover, list, and call:__ A client discovers a server, lists its tools, and calls a tool with JSON-RPC requests that name the protocol version. A wrong request gets a protocol error, while a tool that fails returns a result with its error flag set.
> * __Trust boundaries:__ A server runs code with the permissions of the user, and the language model writes the arguments of each call. The host approves each call, and the server checks every argument and keeps API keys to itself.

With MCP, any language model application that supports this version of the protocol can use our own computational code.
___